# Inferencia — Traductor Inglés → Español (Transformer desde cero)

Este notebook carga el modelo ya entrenado (`transformer_en_es.pth`) y sus vocabularios
(`vocab_en.json`, `vocab_es.json`, `model_config.json`) para traducir texto nuevo, sin volver a entrenar.

**Antes de ejecutar:** sube a este entorno de Colab (panel izquierdo → carpeta → subir archivo) los 4
archivos generados por `mt_en_es_transformer.ipynb`:
- `transformer_en_es.pth`
- `vocab_en.json`
- `vocab_es.json`
- `model_config.json`

(también puedes montar tu Google Drive si los guardaste ahí).

## 1. Imports

In [ ]:
import json, math, re
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Utilidades de texto y vocabulario (mismas que en el entrenamiento)

In [ ]:
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = '<pad>', '<sos>', '<eos>', '<unk>'

def normalize_text(s):
    s = s.strip().lower()
    s = re.sub(r"([.!?¿¡,])", r" \1 ", s)
    s = re.sub(r"[^a-zñáéíóúü¿¡.!?, ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

class Vocab:
    def __init__(self, word2idx):
        self.word2idx = word2idx
        self.idx2word = {i: w for w, i in word2idx.items()}

    def encode(self, sentence, add_sos_eos=True):
        ids = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in sentence.split()]
        if add_sos_eos:
            ids = [self.word2idx[SOS_TOKEN]] + ids + [self.word2idx[EOS_TOKEN]]
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), UNK_TOKEN)
            if w == EOS_TOKEN:
                break
            if w in (SOS_TOKEN, PAD_TOKEN):
                continue
            words.append(w)
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)

## 3. Arquitectura del modelo (idéntica a la del entrenamiento)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.w_q(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.w_o(out)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, mask):
        x = self.embed(src) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, mask)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.embed(tgt) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.fc_out(x)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, n_heads=8, d_ff=512,
                 n_layers=3, dropout=0.1, max_len=100, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.pad_idx = pad_idx

    def make_src_mask(self, src):
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        L = tgt.size(1)
        sub_mask = torch.tril(torch.ones((L, L), device=tgt.device)).bool()
        return pad_mask & sub_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        return self.decoder(tgt, enc_out, src_mask, tgt_mask)

## 4. Cargar vocabularios, configuración y pesos entrenados

In [ ]:
with open('vocab_en.json', encoding='utf-8') as f:
    src_vocab = Vocab(json.load(f))
with open('vocab_es.json', encoding='utf-8') as f:
    tgt_vocab = Vocab(json.load(f))
with open('model_config.json') as f:
    config = json.load(f)

PAD_IDX = src_vocab.word2idx[PAD_TOKEN]

model = Transformer(len(src_vocab), len(tgt_vocab), pad_idx=PAD_IDX, **config).to(device)
model.load_state_dict(torch.load('transformer_en_es.pth', map_location=device))
model.eval()
print('Modelo cargado. Vocabularios:', len(src_vocab), 'EN /', len(tgt_vocab), 'ES')

## 5. Función de traducción (greedy decoding)

In [ ]:
def translate_sentence(sentence, max_len=30):
    model.eval()
    norm = normalize_text(sentence)
    src_ids = torch.tensor([src_vocab.encode(norm)], dtype=torch.long).to(device)
    src_mask = model.make_src_mask(src_ids)
    with torch.no_grad():
        enc_out = model.encoder(src_ids, src_mask)
    tgt_ids = [tgt_vocab.word2idx[SOS_TOKEN]]
    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        tgt_mask = model.make_tgt_mask(tgt_tensor)
        with torch.no_grad():
            out = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)
        next_id = out[0, -1].argmax().item()
        tgt_ids.append(next_id)
        if next_id == tgt_vocab.word2idx[EOS_TOKEN]:
            break
    return tgt_vocab.decode(tgt_ids[1:])

## 6. Probar con oraciones de ejemplo

In [ ]:
ejemplos = [
    'I love you.',
    'What time is it?',
    'The weather is nice today.',
    'Where is the nearest hospital?',
    'I am learning machine learning.',
]
for s in ejemplos:
    print(f'{s!r:40s} -> {translate_sentence(s)}')

## 7. Traducir tu propio texto

In [ ]:
texto = 'Type your own sentence here.'  # <-- cambia esta oración
print(translate_sentence(texto))